# 🌱 GA Potager — atelier data scientist

Workflow du notebook :
1. **Configurer le bac** dans le bloc de code ci-dessous (dimensions, terre, substrats…)
2. **Lancer l'algorithme génétique** (la lib Rust via le CLI en mode `--json`)
3. **Lire le gagnant** et *pourquoi* il gagne (décomposition du score)
4. **Explorer toute la population** : les bonnes propositions, et les moins bonnes

## 1. Configuration du bac

Modifie librement les valeurs, puis réexécute cette cellule (et les suivantes).

In [ ]:
import json
from pathlib import Path

# --- Ton bac, à toi de jouer ---
BAC = {
    "id": "reference",
    "nom": "Bac de référence",
    "dimensions_cm": {"longueur": 120, "largeur": 80, "hauteur": 50},
    "hauteur_terre_cm": 40,
    "marge_bord_cm": 5,
    "protection": "exterieur",
    "substrats": ["terreau", "compost", "fibre_de_coco"],
    "compost_disponible": True,
    "drainage_eau": True,
    "exposition": "plein_soleil",
    "saison_cible": "ete",
    "temp_moyenne_c": {"min": 15, "max": 30},
}

# Où vit le repo (le notebook est dans notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

# Écrit bacs.json : c'est L'entrée de l'AG
(DATA / "bacs.json").write_text(
    json.dumps([BAC], indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"bacs.json écrit : {BAC['nom']} ({BAC['dimensions_cm']['longueur']}x{BAC['dimensions_cm']['largeur']} cm, {BAC['hauteur_terre_cm']} cm de terre)")

## 2. Lancement de l'algorithme génétique

Le binaire Rust est compilé si besoin, puis lancé en mode `--json`. Change `SEED` pour explorer la variabilité des runs.

In [ ]:
import subprocess, sys

SEED = 42  # change pour explorer la variabilité

BIN = ROOT / "ga-plantes" / "target" / "debug" / ("ga-potager-cli.exe" if sys.platform == "win32" else "ga-potager-cli")
if not BIN.exists():
    print("Compilation du binaire...")
    subprocess.run(["cargo", "build", "--manifest-path", str(ROOT / "ga-plantes" / "Cargo.toml")], check=True)

proc = subprocess.run(
    [str(BIN), "--data-dir", str(DATA), "--config-path", str(DATA / "config.json"), "--seed", str(SEED), "--json"],
    capture_output=True, text=True,
)
if proc.returncode != 0:
    print(proc.stderr)  # erreurs de données : la validation explique quoi corriger
    raise SystemExit(1)

resultat = json.loads(proc.stdout[proc.stdout.index("{"):])
print(f"Évolution terminée en {resultat['generations']} générations (seed {resultat['seed']})")

## 3. Le gagnant — et *pourquoi* il gagne

Le score total est la somme de trois composantes pondérées. Le graphe montre laquelle porte la solution.

In [ ]:
import matplotlib.pyplot as plt

meilleur = resultat["meilleur"]
print("🏆 Potager gagnant — bac", meilleur["bac"])
for p in meilleur["plantes"]:
    print(f"   - {p['nom']} ({p['famille']})")
print(f"Score total : {meilleur['score']:.2f}")
print("\nInteractions appliquées :")
for i in meilleur["detail"]["interactions_actives"] or ["(aucune)"]:
    print(f"   {i}")

d = meilleur["detail"]
composantes = {"Rendement (x5)": d["rendement"], "Interactions (x7)": d["interactions"], "Occupation (x10)": d["occupation"]}
couleurs = ["#4caf50" if v >= 0 else "#e53935" for v in composantes.values()]

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(list(composantes.keys()), list(composantes.values()), color=couleurs)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title(f"Pourquoi ce potager gagne — décomposition du score (total {meilleur['score']:.2f})")
for i, v in enumerate(composantes.values()):
    ax.text(v, i, f" {v:+.2f}", va="center")
plt.tight_layout(); plt.show()

## 4. Toute la population — les bonnes et les moins bonnes propositions

L'AG ne produit pas qu'un gagnant : la dernière génération contient toutes les solutions évaluées. L'histogramme montre la distribution, le classement les extrêmes.

In [ ]:
pop = sorted(resultat["population"], key=lambda x: -x["score"])
scores = [x["score"] for x in pop]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(scores, bins=20, color="#66bb6a", edgecolor="white")
ax.axvline(meilleur["score"], color="#1b5e20", linestyle="--", label=f"gagnant ({meilleur['score']:.1f})")
ax.set_xlabel("Score"); ax.set_ylabel("Nombre d'individus")
ax.set_title(f"Distribution des {len(pop)} individus de la population finale")
ax.legend(); plt.tight_layout(); plt.show()

print("🥇 Top 3 :")
for x in pop[:3]:
    print(f"   {x['score']:8.2f}  {' + '.join(x['plantes'])}")
print("\n💀 Flop 3 :")
for x in pop[-3:]:
    print(f"   {x['score']:8.2f}  {' + '.join(x['plantes']) or '(bac vide)'}")

## 5. Comment itérer

- **Change une chose à la fois** : une valeur du bac, la seed, ou une plante dans `plantes.json`
- **Compare les runs** : si le gagnant change du tout au tout entre deux seeds, l'AG explore mal (population trop petite, early stopping trop pressé)
- **Lis le flop** : les mauvaises propositions sont aussi instructives — une compétition pénalisée, un bac quasi vide, un rendement faible
- Les **poids** (5 / 7 / 10) vivent dans `fitness/eval.rs` : changer le `7` en `12` valorise le compagnonnage, puis relance et compare les histogrammes